# Task #38 — Khảo sát thêm đặc trưng domain knowledge chưa xét (Story #7)

Story #6 đã khảo sát tín hiệu theo thời gian, vùng, và hình thức thanh toán. Notebook này khảo sát thêm 2 đặc trưng domain knowledge chưa được xét, đã có sẵn hoặc dễ tính từ dữ liệu thô:

1. **Thời gian từ đặt hàng đến duyệt thanh toán** (`order_purchase_timestamp` → `order_approved_at`) — đã có trong `orders_labeled.csv`, chưa qua xử lý thành đặc trưng.
2. **Tổng khối lượng đơn hàng** (`product_weight_g`, bảng `products`, join qua `order_items`) — chưa từng join vào dataset.

Dùng **Phương án A** (loại nhóm `is_delayed=NA`) làm chính, nhất quán với Story #6, vì đây là input cho mô hình hóa.

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/orders_labeled.csv", low_memory=False)

date_cols = [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

bool_cols = [
    "payment_has_boleto", "payment_has_credit_card", "payment_has_debit_card",
    "payment_has_not_defined", "payment_has_voucher", "items_multi_seller",
]
for col in bool_cols:
    df[col] = df[col].astype("boolean")

df["is_delayed"] = df["is_delayed"].astype("boolean")

df.shape

(99441, 42)

## 1. Thời gian từ đặt hàng đến duyệt thanh toán (`approval_gap_hours`)

In [2]:
df["approval_gap_hours"] = (df["order_approved_at"] - df["order_purchase_timestamp"]).dt.total_seconds() / 3600

print("Tổng số đơn:", len(df))
print("Số đơn thiếu order_approved_at:", df["order_approved_at"].isna().sum())
print("Số đơn approval_gap_hours < 0 (bất thường):", (df["approval_gap_hours"] < 0).sum())

# Phương án A: loại nhóm is_delayed=NA
dfa = df[df["is_delayed"].notna()].copy()
print("\nSố đơn Phương án A (is_delayed xác định):", len(dfa))
print("Trong đó thiếu approval_gap_hours:", dfa["approval_gap_hours"].isna().sum())
print("Trong đó approval_gap_hours < 0:", (dfa["approval_gap_hours"] < 0).sum())

Tổng số đơn: 99441
Số đơn thiếu order_approved_at: 160
Số đơn approval_gap_hours < 0 (bất thường): 0

Số đơn Phương án A (is_delayed xác định): 96476
Trong đó thiếu approval_gap_hours: 14
Trong đó approval_gap_hours < 0: 0


In [3]:
dfa_valid = dfa[dfa["approval_gap_hours"].notna() & (dfa["approval_gap_hours"] >= 0)].copy()
dfa_valid["approval_gap_bucket"] = pd.qcut(dfa_valid["approval_gap_hours"], q=5, duplicates="drop")

gap_result = dfa_valid.groupby("approval_gap_bucket", observed=True).agg(
    n_orders=("is_delayed", "size"),
    delay_rate=("is_delayed", "mean"),
)
gap_result["delay_rate_pct"] = (gap_result["delay_rate"] * 100).round(2)
gap_result

,n_orders,delay_rate,delay_rate_pct
approval_gap_bucket,,,
"(-0.001, 0.201]",19385,0.068971,6.9
"(0.201, 0.272]",19223,0.072309,7.23
"(0.272, 0.61]",19269,0.079558,7.96
"(0.61, 21.194]",19292,0.087549,8.75
"(21.194, 741.444]",19293,0.097341,9.73


## 2. Tổng khối lượng đơn hàng (`items_total_weight_g`)

Chưa có trong `orders_labeled.csv` — join `order_items` với `products` để lấy `product_weight_g`, cộng dồn theo `order_id`.

In [4]:
items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")

items_weight = items.merge(products[["product_id", "product_weight_g"]], on="product_id", how="left")
order_weight = items_weight.groupby("order_id")["product_weight_g"].sum(min_count=1).rename("items_total_weight_g")

print("Số order_id trong order_items:", items["order_id"].nunique())
print("Số order_id có tổng weight NaN (thiếu product_weight_g cho toàn bộ item):", order_weight.isna().sum())

dfa_w = dfa.merge(order_weight, left_on="order_id", right_index=True, how="left")
print("Số đơn Phương án A thiếu items_total_weight_g:", dfa_w["items_total_weight_g"].isna().sum())

Số order_id trong order_items: 98666
Số order_id có tổng weight NaN (thiếu product_weight_g cho toàn bộ item): 16
Số đơn Phương án A thiếu items_total_weight_g: 16


In [5]:
dfa_w_valid = dfa_w[dfa_w["items_total_weight_g"].notna()].copy()
dfa_w_valid["weight_bucket"] = pd.qcut(dfa_w_valid["items_total_weight_g"], q=5, duplicates="drop")

weight_result = dfa_w_valid.groupby("weight_bucket", observed=True).agg(
    n_orders=("is_delayed", "size"),
    delay_rate=("is_delayed", "mean"),
)
weight_result["delay_rate_pct"] = (weight_result["delay_rate"] * 100).round(2)
weight_result

,n_orders,delay_rate,delay_rate_pct
weight_bucket,,,
"(-0.001, 250.0]",20498,0.07391,7.39
"(250.0, 500.0]",18333,0.084056,8.41
"(500.0, 1100.0]",19056,0.076039,7.6
"(1100.0, 2800.0]",19422,0.083359,8.34
"(2800.0, 184400.0]",19151,0.088925,8.89


## 3. Tổng kết tín hiệu

| Đặc trưng | Tỉ lệ trễ theo nhóm (ngũ phân vị) | Biên độ | Đánh giá tín hiệu |
|---|---|---|---|
| `approval_gap_hours` | 6,90% → 7,23% → 7,96% → 8,75% → 9,73% | ~2,8 điểm %, tăng đơn điệu theo nhóm | Trung bình — rõ hơn hình thức thanh toán (7,01–8,88%, Story #6), nhưng yếu hơn nhiều so với vùng (chênh ~4 lần, Story #6) |
| `items_total_weight_g` | 7,39% → 8,41% → 7,60% → 8,34% → 8,89% | ~1,5 điểm %, không đơn điệu (nhóm 3 giảm rồi tăng lại) | Yếu — tương đương mức tín hiệu của hình thức thanh toán |

Dữ liệu thiếu không đáng kể ở cả 2 đặc trưng (14/96.476 và 16/96.476 đơn Phương án A, đều dưới 0,02%).